# Beyond ReAct [Step 08.01 - Where the loop breaks down]

> **MLCourse - Agentic AI - LangGraph**

In `02_tool_using_agents/02_react_agent_loop` you built the pattern that powers
most agent demos on the internet:

```
loop:
    THINK   -> the model reasons about what to do next
    ACT     -> it calls a tool
    OBSERVE -> the tool result is appended to the conversation
until the model answers instead of calling a tool
```

ReAct is a genuinely good default: simple, two nodes, handles a huge range of tasks.
But it has structural weaknesses, and this notebook **measures** them on a real run
rather than asserting them.

### What you'll learn

- How to **instrument** an agent: tokens, LLM calls, tool calls, latency, correctness.
- Weakness 1 - **no global plan**: greedy and myopic, one step at a time.
- Weakness 2 - **no learning from failure**: a retry repeats the same mistake.
- Weakness 3 - **quadratic token waste**: the whole transcript is resent every step.
- Weakness 4 - **no separation of concerns**: one model call plans, acts and answers.

### Key takeaways

- ReAct's input cost grows roughly with the **square** of the step count, because
  step *k* resends everything from steps 1..k-1.
- ReAct has no memory across attempts.
- Reflexion, Plan-and-Execute and ReWOO each target one of these weaknesses.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                  # environment variable access
import time                                # timing + backoff sleeps
from pathlib import Path                   # locating the track root
from dotenv import load_dotenv             # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we find the track root `03_agentic_ai`,
# then load the (gitignored) .env that lives there. Every provider-touching
# notebook in this track uses exactly this block.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

GROQ_KEY = os.getenv("GROQ_API_KEY")       # never print this value
GROQ_MODEL = "qwen/qwen3.8-27b"            # fast hosted model, generous free tier
OLLAMA_MODEL = "llama3.1:8b"               # local fallback if Groq is unavailable


def make_llm(temperature: float = 0.0, max_tokens: int = 512):
    """Return a chat model. Groq first (fast, hosted); local Ollama as fallback.

    OpenAI is never used anywhere in this course.
    """
    if GROQ_KEY:
        from langchain_groq import ChatGroq
        return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                        temperature=temperature, max_tokens=max_tokens)
    from langchain_ollama import ChatOllama
    return ChatOllama(model=OLLAMA_MODEL, temperature=temperature)


def safe_invoke(model, messages, retries: int = 4, pause: float = 1.5):
    """Invoke a chat model with exponential backoff on rate limits (HTTP 429).

    Groq's free tier allows roughly 8000 tokens per minute. Teaching notebooks
    fire many small calls in a row, so a retry loop is not optional here.
    """
    delay = pause
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(pause)              # pace the next call politely
            return out
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print("  [backoff] %s -- retrying in %.1fs" % (type(exc).__name__, delay))
            time.sleep(delay)
            delay *= 2                     # exponential backoff
    raise RuntimeError("unreachable")


print("Track root :", TRACK.name)
print("Provider   :", "Groq / " + GROQ_MODEL if GROQ_KEY else "Ollama / " + OLLAMA_MODEL)


### The shared task world


In [ ]:
# Every notebook in this module attacks THE SAME task with a different reasoning
# pattern, so the comparison in notebook 05 is apples-to-apples.

from langchain_core.tools import tool

# A tiny deterministic "database". Deterministic matters: we need to check
# correctness automatically, without a human reading the answer.
POPULATION = {"tokyo": 13_960_000, "lagos": 15_400_000, "lima": 9_750_000}
AREA_KM2 = {"tokyo": 2194, "lagos": 1171, "lima": 2672}

TOOL_CALLS = {"count": 0}          # instrumentation: how many tool calls happened


@tool
def population(city: str) -> str:
    """Return the population of a city as a plain number string.

    Args:
        city: City name, e.g. "Tokyo".
    """
    TOOL_CALLS["count"] += 1
    return str(POPULATION.get(city.strip().lower(), "unknown city"))


@tool
def area_km2(city: str) -> str:
    """Return the land area of a city in square kilometres as a plain number string.

    Args:
        city: City name, e.g. "Tokyo".
    """
    TOOL_CALLS["count"] += 1
    return str(AREA_KM2.get(city.strip().lower(), "unknown city"))


TOOLS = [population, area_km2]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

TASK = (
    "Among Tokyo, Lagos and Lima, which city has the highest population density "
    "(people per square kilometre)? Answer with the city name and the density "
    "rounded to the nearest whole number."
)

# Ground truth, computed here so the notebook can grade itself.
DENSITIES = {c: POPULATION[c] / AREA_KM2[c] for c in POPULATION}
GT_CITY = max(DENSITIES, key=DENSITIES.get)
GT_DENSITY = round(DENSITIES[GT_CITY])

print("Task:", TASK)
print()
for c in sorted(DENSITIES, key=DENSITIES.get, reverse=True):
    print("  %-6s %9d / %5d = %7.0f people/km2" % (c, POPULATION[c], AREA_KM2[c], DENSITIES[c]))
print()
print("Ground truth -> %s, %d" % (GT_CITY.title(), GT_DENSITY))


def grade(answer: str) -> bool:
    """Automatic grader: the answer must name the right city AND the right density.

    The density is accepted within +/-2 to tolerate rounding differences.
    """
    import re
    if not answer:
        return False
    low = answer.lower()
    if GT_CITY not in low:
        return False
    cleaned = low.replace(",", "").replace(".", " ")
    numbers = [int(n) for n in re.findall(r"\d+", cleaned)]
    return any(abs(n - GT_DENSITY) <= 2 for n in numbers)


### Instrumentation: a token and latency meter


In [ ]:
import time


class Meter:
    """Accumulates token usage, call counts and wall-clock time for one run.

    Every pattern in this module is wrapped in one of these, so notebook 05 can
    compare them on identical instrumentation.
    """

    def __init__(self, name):
        self.name = name
        self.input_tokens = 0
        self.output_tokens = 0
        self.llm_calls = 0
        self.tool_calls = 0
        self.seconds = 0.0
        self._t0 = None

    def start(self):
        TOOL_CALLS["count"] = 0
        self._t0 = time.time()
        return self

    def stop(self):
        self.seconds = time.time() - self._t0
        self.tool_calls = TOOL_CALLS["count"]
        return self

    def record(self, message):
        """Add one AIMessage's usage to the totals, then return the message."""
        usage = getattr(message, "usage_metadata", None) or {}
        if usage:
            self.input_tokens += usage.get("input_tokens", 0)
            self.output_tokens += usage.get("output_tokens", 0)
            self.llm_calls += 1
        return message

    def record_all(self, messages):
        """Add usage from every AIMessage in a list (for create_agent results)."""
        for m in messages:
            if getattr(m, "usage_metadata", None):
                self.record(m)
        return messages

    @property
    def total_tokens(self):
        return self.input_tokens + self.output_tokens

    def report(self, answer=None, correct=None):
        print()
        print("=" * 62)
        print("PATTERN : %s" % self.name)
        print("-" * 62)
        print("LLM calls    : %d" % self.llm_calls)
        print("tool calls   : %d" % self.tool_calls)
        print("input tokens : %d" % self.input_tokens)
        print("output tokens: %d" % self.output_tokens)
        print("TOTAL tokens : %d" % self.total_tokens)
        print("latency      : %.1fs" % self.seconds)
        if correct is not None:
            print("correct      : %s" % ("YES" if correct else "NO"))
        print("=" * 62)
        if answer:
            print(answer)
        return self


### 1. A ReAct agent on the shared task

`create_agent` builds the standard ReAct loop for us.

> **API note:** as of LangChain v1 / LangGraph v1, `create_agent` lives in
> **`langchain.agents`**, not `langgraph.prebuilt`, and the old `prompt=` argument
> is now `system_prompt=`. If you find older tutorials importing it from
> `langgraph.prebuilt`, they predate the move.

In [4]:
from langchain.agents import create_agent

SYS = ("You are a careful analyst. Use the provided tools to look up facts. "
       "Do the arithmetic yourself. Give a short final answer.")

react_agent = create_agent(model=make_llm(max_tokens=400), tools=TOOLS, system_prompt=SYS)

m_react = Meter("ReAct").start()
result = react_agent.invoke({"messages": [("user", TASK)]})
m_react.stop()
m_react.record_all(result["messages"])

react_answer = result["messages"][-1].content
m_react.report(react_answer, grade(react_answer))


PATTERN : ReAct
--------------------------------------------------------------
LLM calls    : 3
tool calls   : 7
input tokens : 1874
output tokens: 309
TOTAL tokens : 2183
latency      : 1.3s
correct      : YES
Tokyo: 13,960,000 / 2,194 ≈ 6,363
Lagos: 15,400,000 / 1,171 ≈ 13,151
Lima: 9,750,000 / 2,672 ≈ 3,649

Lagos has the highest population density.

**Lagos, ≈ 13,151 people/km²**


### 2. Look at the actual trace

The message list *is* the reasoning trace. Print it and the weaknesses stop being
theoretical.

In [5]:
for i, msg in enumerate(result["messages"]):
    kind = type(msg).__name__.replace("Message", "")
    if getattr(msg, "tool_calls", None):
        calls = ", ".join("%s(%s)" % (tc["name"], tc["args"]) for tc in msg.tool_calls)
        print("%2d %-9s -> CALLS %s" % (i, kind, calls[:95]))
    else:
        print("%2d %-9s    %s" % (i, kind, str(msg.content).replace("\n", " ")[:90]))

 0 Human        Among Tokyo, Lagos and Lima, which city has the highest population density (people per squ
 1 AI        -> CALLS population({'city': 'Tokyo'}), population({'city': 'Lagos'}), population({'city': 'Lima'}), are
 2 Tool         13960000
 3 Tool         15400000
 4 Tool         9750000
 5 Tool         2194
 6 Tool         1171
 7 Tool         2672
 8 AI        -> CALLS population({'city': 'Tokyo'})
 9 Tool         13960000
10 AI           Tokyo: 13,960,000 / 2,194 ≈ 6,363 Lagos: 15,400,000 / 1,171 ≈ 13,151 Lima: 9,750,000 / 2,6


### 3. Weakness 1 - no global plan

ReAct decides **one step at a time**. At step 3 it has made no commitment about
steps 5 and 6. Consequences:

- **It cannot parallelise.** Our task needs six independent lookups. Nothing about
  them depends on each other, yet each decision waits for the previous observation.
- **It can wander.** With no plan to check progress against, a distracted model has
  nothing to snap it back.
- **You cannot review the plan before it runs.** There is no artifact to approve,
  which matters for the human-in-the-loop patterns from module 04.

In [6]:
print("Facts required :", len(POPULATION) * 2, "(3 cities x 2 attributes)")
print("Truly dependent:", 0, "- every lookup could run in parallel")
print("Tool calls made:", m_react.tool_calls)
print("LLM round trips:", m_react.llm_calls)
print()
print("Each round trip is a network call whose result the NEXT one waits for.")
print("A pattern that knows all six lookups up front can batch them (see 04_rewoo).")

Facts required : 6 (3 cities x 2 attributes)
Truly dependent: 0 - every lookup could run in parallel
Tool calls made: 7
LLM round trips: 3

Each round trip is a network call whose result the NEXT one waits for.
A pattern that knows all six lookups up front can batch them (see 04_rewoo).


### 4. Weakness 2 - no learning from failure

The sharp version of the problem: give the agent a task it usually gets wrong, then
retry. In plain ReAct, "retry" means running the same thing again from a clean
slate - attempt 2 is statistically identical to attempt 1.

In [7]:
tricky = ("Using ONLY the tools, compute the population density of Lagos and express "
          "it as people per HECTARE, not per square kilometre. "
          "Give just the number, rounded to one decimal place.")

expected = round(POPULATION["lagos"] / (AREA_KM2["lagos"] * 100), 1)
print("expected:", expected, "people/hectare")
print()

for n in (1, 2):
    r = react_agent.invoke({"messages": [("user", tricky)]})
    ans = r["messages"][-1].content.strip().replace("\n", " ")
    print("attempt %d: %s" % (n, ans[:150]))
    time.sleep(3)                      # pace against the free-tier rate limit

expected: 131.5 people/hectare



attempt 1: Population = 15,400,000; Area = 1,171 km² = 117,100 hectares.  Density = 15,400,000 / 117,100 ≈ 131.5 people/ha.  131.5


attempt 2: Population = 15,400,000; Area = 1,171 km² = 117,100 hectares.  Density = 15,400,000 / 117,100 ≈ 131.5 people/ha.  131.5


In [8]:
print("Attempt 2 received ZERO information about attempt 1.")
print("Same prompt, same tools, same model -> the same failure mode recurs.")
print()
print("Reflexion (notebook 02) fixes exactly this: it writes a natural-language")
print("lesson from the failure and feeds that lesson into the retry, so attempts")
print("accumulate as memory instead of resetting.")

Attempt 2 received ZERO information about attempt 1.
Same prompt, same tools, same model -> the same failure mode recurs.

Reflexion (notebook 02) fixes exactly this: it writes a natural-language
lesson from the failure and feeds that lesson into the retry, so attempts
accumulate as memory instead of resetting.


### 5. Weakness 3 - quadratic token waste

This is the one people underestimate. On every step, ReAct resends the **entire**
message list. Step 1 sends the system prompt plus the question. Step 2 sends all of
that plus the first thought, tool call and observation. Step 3 sends all of *that*
plus more.

If each step adds roughly a constant amount of text, total input tokens over N steps
grow with N-squared, not N. Here is the real growth in the run above.

In [9]:
per_call = [(m.usage_metadata["input_tokens"], m.usage_metadata["output_tokens"])
            for m in result["messages"] if getattr(m, "usage_metadata", None)]

print(" call | input tokens | output tokens | growth vs previous")
print("------+--------------+---------------+-------------------")
prev = None
for n, (inp, outp) in enumerate(per_call, start=1):
    growth = "" if prev is None else "+%d" % (inp - prev)
    print("  %2d  |    %6d    |     %5d     | %s" % (n, inp, outp, growth))
    prev = inp

total_in = sum(p[0] for p in per_call)
first_in = per_call[0][0]
flat = first_in * len(per_call)
print()
print("input tokens on the FIRST call : %d" % first_in)
print("input tokens on the LAST call  : %d" % per_call[-1][0])
print("total input tokens billed      : %d" % total_in)
print("if context never re-grew       : %d  (%d calls x first-call size)"
      % (flat, len(per_call)))
print("overhead from resending history: %d tokens (%.0f%% extra)"
      % (total_in - flat, 100 * (total_in - flat) / max(1, flat)))

 call | input tokens | output tokens | growth vs previous
------+--------------+---------------+-------------------
   1  |       444    |       168     | 
   2  |       689    |        27     | +245
   3  |       741    |       114     | +52

input tokens on the FIRST call : 444
input tokens on the LAST call  : 741
total input tokens billed      : 1874
if context never re-grew       : 1332  (3 calls x first-call size)
overhead from resending history: 542 tokens (41% extra)


> **Why this matters in production:** input tokens are billed, and long contexts are
> slower to process. A 20-step ReAct trajectory can spend more on *re-reading its own
> history* than on doing the work. ReWOO (notebook 04) attacks this directly by never
> putting tool observations in front of the reasoning model at all.

### 6. Weakness 4 - no separation of concerns

In ReAct, one model call decides *what to do*, *how to do it*, and *what the answer
is*. That means:

- You cannot use a cheap model for the mechanical parts and an expensive one for planning.
- You cannot cache the plan - there is no plan object.
- You cannot show a user the plan for approval before spending money on it.

Plan-and-Execute (notebook 03) and ReWOO (notebook 04) both split these roles apart.

### 7. The map of this module

| Pattern | Fixes | Core idea | Notebook |
|---------|-------|-----------|----------|
| **ReAct** (baseline) | - | Think / act / observe until done | `02_tool_using_agents/02` |
| **Reflexion** | No learning from failure | Actor -> evaluator -> self-reflection, carried into the retry | [02](02_reflexion.ipynb) |
| **Plan-and-Execute** | No global plan | Planner writes all steps; executor runs them; replanner adapts | [03](03_plan_and_execute.ipynb) |
| **ReWOO** | Token waste + no plan | Plan with variables, batch tool calls, single solver pass | [04](04_rewoo.ipynb) |
| **Comparison** | - | All four on this exact task, measured | [05](05_choosing_a_pattern.ipynb) |

### 8. The baseline numbers


In [10]:
print("BASELINE (ReAct) on the shared task")
print("  correct      :", grade(react_answer))
print("  LLM calls    :", m_react.llm_calls)
print("  tool calls   :", m_react.tool_calls)
print("  total tokens :", m_react.total_tokens)
print("  latency      : %.1fs" % m_react.seconds)
print()
print("Every other pattern in this module is measured against these numbers.")

BASELINE (ReAct) on the shared task
  correct      : True
  LLM calls    : 3
  tool calls   : 7
  total tokens : 2183
  latency      : 1.3s

Every other pattern in this module is measured against these numbers.


### Recap

- ReAct is greedy: **no global plan**, so no parallelism, no reviewable artifact,
  no progress check.
- ReAct has **no cross-attempt memory**: a retry repeats the mistake.
- ReAct's input cost grows **quadratically** because it resends its whole transcript.
- ReAct **fuses** planning, acting and answering into one call.

None of this makes ReAct bad. It makes it a default, not a universal answer.

### Next

**[02_reflexion](02_reflexion.ipynb)** - make failure informative.